<a href="https://colab.research.google.com/github/Git-Hub-Ran/rag-qa-langchain/blob/Dev/rag-qa-langchain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install "langchain==0.3.27" "langchain-community>=0.3.0" "langchain-core>=0.3.0" "langchain-openai>=0.3.0" langchain-text-splitters chromadb unstructured python-dotenv

In [119]:
#Connect to Google Drive to use API key:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [120]:
#Loading API key from .env:
from dotenv import load_dotenv
import os

load_dotenv("/content/drive/MyDrive/Colab Notebooks/openai-chatbot/.env")

True

In [121]:
from langchain_community.document_loaders import UnstructuredURLLoader

# Webites that will be saved as documents:
urls = [
    "https://www.governor.ny.gov/news", #New York state news website
    "https://en.wikipedia.org/wiki/Chocolate_chip_cookie?utm_source=chatgpt.com" #Chocolate chip cookie
]

#load documents:
loader = UnstructuredURLLoader(urls=urls)
docs = loader.load()

print(f"Number of documents that were loaded: {len(docs)}")

Number of documents that were loaded: 2


In [122]:
docs[0].page_content[:5000]  # Validation that the document was loaded : see the content of the document

"Pressroom\n\nOfficial news from the Office of the Governor\n\nFebruary 16, 2026\n\nInvesting in New York City\n\nGovernor Hochul and Mayor Mamdani announced an additional $1.5 billion in funding to help New York City address fiscal challenges, including $510 million in recurring funding.\n\nPress Release\n\nFeatured News\n\nInvesting in Continued Preservation of Historical Sites\n\nInvesting in Continued Preservation of Historical Sites\n\nFebruary 15, 2026 | 3:41 PM EST\n\nGovernor Hochul announced $1.2 million for historical societies across New York State.\n\n$3.8M Grant Program Highlights Black History Across NYS\n\n$3.8M Grant Program Highlights Black History Across NYS\n\nFebruary 15, 2026 | 2:33 PM EST\n\nGovernor Hochul announced a $3.8 million new grant program to support and promote the history and achievements of African Americans and people of African descent throughout the State.\n\nStatement from Governor Kathy Hochul\n\nStatement from Governor Kathy Hochul\n\nFebruary 1

In [123]:
#Break the docs into smaller chunks:
from langchain_text_splitters import RecursiveCharacterTextSplitter #This text splitter is the recommended one for generic text.

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,#The maximum size of a chunk
    chunk_overlap=100 #Target overlap between chunks. Overlapping chunks helps to mitigate loss of information when context is divided between chunks.
    )
chunks = text_splitter.split_documents(docs)

print(f"Number of chunks: {len(chunks)}")

Number of chunks: 7


In [124]:
#create embedding:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings()

#creating vector store with Chroma:
from langchain_community.vectorstores import Chroma

vector_store = Chroma.from_documents(
    documents=chunks, #every chunk in the list
    embedding=embeddings #the function that convert every chunk into a numeric vector
)
#Now the vector store is ready for semantic search

In [125]:
#Create a retriever from the vector store. The retriever can get a question and return an answer from the relevent chunks of documents:
retriever = vector_store.as_retriever( search_kwargs={"k": 1})

In [126]:
from langchain_openai import ChatOpenAI

#Creating LLM:
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
    api_key=os.getenv("OPENAI_API_KEY")
)

In [130]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate


#The question I want to ask :
queries = [
    "What are the main news topics on the New York governor website?",
    "Who invented the chocolate chip cookie and where?"
]

#define the prompt:
prompt = ChatPromptTemplate.from_messages([
   ("system",
          "You are a helpful assistant. Answer the user's question based on the context below. "
          ".\n\n{context}"),
    ("human", "{input}")
])

# Create the answer-generation chain (LLM + prompt that processes retrieved documents):
question_answer_chain = create_stuff_documents_chain(llm, prompt)

# Build complete retrieval augmented generation (RAG) pipeline:
qa_chain = create_retrieval_chain(retriever, question_answer_chain)

#get an answer for each question:
for query in queries:
    result = qa_chain.invoke({"input": query})
    print(f"Q: {query}")
    print(f"A: {result['answer']}\n")

Q: What are the main news topics on the New York governor website?
A: The main news topics on the New York governor's website include investments in New York City to address fiscal challenges, as highlighted by the announcement of an additional $1.5 billion in funding. Another key topic is the continued preservation of historical sites, which was featured in a recent press release.

Q: Who invented the chocolate chip cookie and where?
A: The chocolate chip cookie was invented by Ruth Wakefield in 1938 at the Toll House Inn in Whitman, Massachusetts. She created the cookie by adding chopped chocolate to her butter cookie recipe, which eventually became known as the Toll House cookie.



In [129]:
#Chacking that the bot returning corrct answer:
docs = retriever.get_relevant_documents(queries[0])
print("Retrieved document snippet:", docs[0].page_content[:1000])


Retrieved document snippet: Pressroom

Official news from the Office of the Governor

February 16, 2026

Investing in New York City

Governor Hochul and Mayor Mamdani announced an additional $1.5 billion in funding to help New York City address fiscal challenges, including $510 million in recurring funding.

Press Release

Featured News

Investing in Continued Preservation of Historical Sites

Investing in Continued Preservation of Historical Sites

February 15, 2026 | 3:41 PM EST
